# Tune NN, LR, SVC, LightGBM, XGBoost

In [ ]:
import sys
import json

sys.path.append("../")
from src.config import BASE_PATH
from src.swarm import run_swarms

Set globals

In [ ]:
N_TRIALS = 250
MODEL_ABRV_LIST = [
    "nn",
    "lr",
    "lgbm",
    "svc",
    "xgb",
]
OUTCOME_LIST = [
    "any",
    "serious",
    "ssi",
    "pneumo",
    "bleed",
    "reoper",
]
SCORING = "average_precision"
DATA_IMP_DIR = BASE_PATH / "data" / "processed_reduced"
N_THREADS = 2
N_CV_JOBS = 5
CMD_DIR = BASE_PATH / "swarm_dir" / "commands"

In [ ]:
def make_cmd(
    model_abrv,
    outcome_name,
    scoring,
    log_file_path,
    import_dir,
    n_trials,
    n_threads,
    n_cv_jobs,
    model_save_dir,
):
    cmd_str = f"export PYTHONPATH={BASE_PATH}; \
			export OMP_NUM_THREADS={n_threads}; \
			uv run python -m src.tune \
			--model_abrv {model_abrv} \
			--outcome_name {outcome_name}\
			--scoring {scoring}\
			--log_file_path {log_file_path} \
			--import_dir {import_dir} \
            --model_save_dir {model_save_dir}\
    		--n_trials {n_trials}\
    		--cv_jobs {n_cv_jobs}"
    return " ".join(cmd_str.split())

In [ ]:
full_cmd_list = []
for model_abrv in MODEL_ABRV_LIST:
    swarm_path = CMD_DIR / f"{model_abrv}.swarm"
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    if swarm_path.exists():
        swarm_path.unlink()
    model_outcome_list = []
    for outcome in OUTCOME_LIST:
        log_file_path = BASE_PATH / "logs" / "tune" / model_abrv / f"{outcome}.log"
        model_save_dir = BASE_PATH / "models" / "trained" / outcome
        cmd_str = make_cmd(
            model_abrv=model_abrv,
            outcome_name=f"outcome_{outcome}",
            scoring=SCORING,
            log_file_path=log_file_path,
            import_dir=DATA_IMP_DIR,
            n_trials=N_TRIALS,
            n_threads=N_THREADS,
            n_cv_jobs=N_CV_JOBS,
            model_save_dir=model_save_dir,
        )
        model_outcome_list.append(cmd_str)
    swarm_path.write_text("\n".join(model_outcome_list))
    full_cmd_list += model_outcome_list
len(full_cmd_list)

Run local sequentially

In [ ]:
# import subprocess

# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     subprocess.run(cmd, shell=True, check=True)

Run swarms

In [ ]:
run_swarms(
    model_list=MODEL_ABRV_LIST,
    swarm_log_dir=BASE_PATH / "swarm_dir" / "logs",
    cmd_dir=CMD_DIR,
    n_jobs=N_CV_JOBS,
    n_threads=N_THREADS,
    n_trials=N_TRIALS,
)